In [0]:
%sql
-- null fill in can start from here
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_phase3_step1;
create or replace table dev.mohit_gangwani.smoothing_nulls_phase3_step1 as 
with vcf as (
  select 
    sum(new_group) over (partition by fk_tvid order by session_start) group_id,
    *
  from (
    select 
      a.fk_tvid,
      a.fk_show_id,
      a.fk_station_id,
      a.session_start,
      a.session_end,
      a.session_duration,
      a.media_time_start,
      a.media_time_end,
      a.runtime,
      a.fk_input_source_id,
      a.fk_content_id,
      a.is_live,
      -- a.fk_schedule_id,
      a.airdate,
      a.tms_show_id,
      a.tms_airdate,
      a.tms_station_id,
      a.vizio_epg_station IS NOT NULL AS non_null_vizio_epg,
      lag(a.fk_station_id) over (partition by a.fk_tvid order by a.session_start) AS prev_fk_st_id,
      case 
        when a.fk_station_id is null 
        and lag(a.fk_station_id) over (partition by a.fk_tvid order by a.session_start) is null
        and a.fk_input_source_id = lag(a.fk_input_source_id) over (partition by a.fk_tvid order by a.session_start)
        and datediff(second, lag(a.session_end) over (partition by a.fk_tvid order by a.session_start), a.session_start) <= 1
        -- only consecutive nulls merged, sess_gap <= 1
         then 0
        else 1
    end AS new_group
    from dev.mohit_gangwani.smoothing_nulls_comm_part_optimized a
    -- WHERE session_start >= CURRENT_DATE - 1
    --   AND session_start < CURRENT_DATE
    --   AND session_end > session_start
    order by fk_tvid, session_start
    )
)
select *,
lead(fk_input_source_id) over (partition by fk_tvid order by session_start) next_input,
lag(fk_input_source_id) over (partition by fk_tvid order by session_start) prev_input,

lead(fk_show_id) over (partition by fk_tvid order by session_start) next_show_id,
lag(fk_show_id) over (partition by fk_tvid order by session_start) prev_show_id,

lead(fk_station_id) over (partition by fk_tvid order by session_start) next_st_id,
lag(fk_station_id) over (partition by fk_tvid order by session_start) prev_st_id,

-- lead(fk_schedule_id) over (partition by fk_tvid order by session_start) next_sche_id,
-- lag(fk_schedule_id) over (partition by fk_tvid order by session_start) prev_sche_id,

lead(airdate) over (partition by fk_tvid order by session_start) next_airdate,
lag(airdate) over (partition by fk_tvid order by session_start) prev_airdate,

lead(sess_dur) over (partition by fk_tvid order by session_start) next_sess_dur,
lag(sess_dur) over (partition by fk_tvid order by session_start) prev_sess_dur,

lead(is_live) over (partition by fk_tvid order by session_start) next_is_live,
lag(is_live) over (partition by fk_tvid order by session_start) prev_is_live,

lead(media_time_start) over (partition by fk_tvid order by session_start) next_media_time_start,
lag(media_time_start) over (partition by fk_tvid order by session_start) prev_media_time_start,  

lead(media_time_end) over (partition by fk_tvid order by session_start) next_media_time_end,
lag(media_time_end) over (partition by fk_tvid order by session_start) prev_media_time_end,

lead(runtime) over (partition by fk_tvid order by session_start) next_runtime,
lag(runtime) over (partition by fk_tvid order by session_start) prev_runtime,

lead(fk_content_id) over (partition by fk_tvid order by session_start) next_content,
lag(fk_content_id) over (partition by fk_tvid order by session_start) prev_content,

lead(session_start) over (partition by fk_tvid order by session_start) next_start,
lag(session_end) over (partition by fk_tvid order by session_start) prev_end,

lead(tms_airdate) over (partition by fk_tvid order by session_start) next_tms_airdate,
lag(tms_airdate) over (partition by fk_tvid order by session_start) prev_tms_airdate,

lead(tms_station_id) over (partition by fk_tvid order by session_start) next_tms_st_id,
lag(tms_station_id) over (partition by fk_tvid order by session_start) prev_tms_st_id,    

lead(tms_show_id) over (partition by fk_tvid order by session_start) next_tms_show_id,
lag(tms_show_id) over (partition by fk_tvid order by session_start) prev_tms_show_id
from (
    select 
        group_id,
        fk_tvid,
        max(fk_station_id) fk_station_id,
        max(fk_input_source_id) fk_input_source_id, 
        -- max(fk_schedule_id) fk_schedule_id,
        max(airdate) airdate,
        max(fk_show_id) fk_show_id,
        max(runtime) runtime,
        max(fk_content_id) fk_content_id,
        min(session_start) session_start,
        max(session_end) session_end,
        max(is_live) is_live,
        min(media_time_start) media_time_start,
        max(media_time_end) media_time_end,
        max(tms_airdate) tms_airdate,
        max(tms_show_id) tms_show_id,
        max(tms_station_id) tms_station_id,
        sum(session_duration) sess_dur,
        count(*) n_sess
    from vcf
    group by 1,2
)

In [0]:
# %sql
# SELECT *
# FROM dev.mohit_gangwani.smoothing_nulls_phase3_step2
# ORDER BY fk_tvid, session_start
# LIMIT 1000

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_phase3_step2;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_phase3_step2 as
WITH vcf AS (
    SELECT c.*
    , CASE WHEN c.prev_st_id is not null and c.next_st_id is not null and c.prev_st_id <> c.next_st_id THEN 'Diff'
           WHEN c.prev_st_id = c.next_st_id THEN 'Same'
      END AS station_check
    , CASE WHEN c.prev_is_live = FALSE and c.next_is_live = FALSE and c.sess_dur <= 300 THEN 'TS'
           WHEN c.prev_is_live = TRUE and c.next_is_live = TRUE and c.sess_dur <= 180 THEN 'Live'
      END AS prevnextlive_sessdur_check
    , timestampdiff(second, c.prev_end, c.session_start) <= 2 and timestampdiff(second, c.session_end, c.next_start) <= 2 AS prev_next_diff_2_second
    , CASE WHEN c.prev_show_id = c.next_show_id THEN 'Same'
           WHEN c.prev_show_id <> c.next_show_id THEN 'Diff'
      END AS show_id_check
    FROM dev.mohit_gangwani.smoothing_nulls_phase3_step1 c
    WHERE c.fk_station_id IS NULL
      AND c.prev_input = next_input
      AND c.prev_input = fk_input_source_id
      and not (date_trunc('hour', c.session_start) + interval '1 hour' = date_trunc('hour', c.session_end)
              or (minute(c.session_end) = 59 and second(c.session_end) = 59)
              or (minute(c.session_start) = 00 and second(c.session_start) = 00)
    )
)
SELECT c.*, 
    CASE
      -- === Same-station ===
      WHEN station_check = 'Same' AND prevnextlive_sessdur_check IN ('TS', 'Live') AND c.prev_next_diff_2_second THEN
        CASE
            WHEN c.prevnextlive_sessdur_check = 'Live'
                 AND c.show_id_check = 'Same'
                 AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                 AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
            THEN 7  -- live / same-show / fillable from prev
            WHEN c.prevnextlive_sessdur_check = 'Live'
                 AND c.show_id_check = 'Diff'
                 AND c.next_media_time_start - c.sess_dur >= 0
                 AND c.next_media_time_start <= c.next_runtime
            THEN 4  -- live / diff-show / fillable from next
            WHEN c.prevnextlive_sessdur_check = 'Live'
                 AND c.show_id_check = 'Diff'
                 AND c.next_media_time_start > 0
                 AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                 AND c.next_media_time_start <= c.next_runtime
            THEN 5  -- live / diff-show / fillable partially next + prev
            WHEN c.prevnextlive_sessdur_check = 'Live'
                 AND c.show_id_check = 'Diff'
                 AND c.prev_media_time_end + c.sess_dur <= prev_runtime
            THEN 6  -- live / diff-show / fillable from prev
            WHEN c.prevnextlive_sessdur_check = 'TS'
                 AND c.show_id_check = 'Same'
                 AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                 AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
            THEN 14 -- ts / same-show / fast-forwarding / fillable from prev
            WHEN c.prevnextlive_sessdur_check = 'TS'
                 AND c.show_id_check = 'Same'
                 AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                 AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
            THEN 16 -- ts / same-show / rewinding / fillable from prev
            WHEN c.prevnextlive_sessdur_check = 'TS'
                 AND c.show_id_check = 'Same'
                 AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                 AND c.next_media_time_start - c.sess_dur >= 0
                 AND c.next_media_time_start <= c.next_runtime
            THEN 15 -- ts / same-show / rewinding / fillable from next
            WHEN c.prevnextlive_sessdur_check = 'TS'
                 AND c.show_id_check = 'Diff'
                 AND c.next_media_time_start - c.sess_dur >= 0
                 AND c.next_media_time_start <= c.next_runtime
            THEN 11 -- ts / diff-show / fillable from next
            WHEN c.prevnextlive_sessdur_check = 'TS'
                 AND c.show_id_check = 'Diff'
                 AND c.next_media_time_start > 0
                 AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= prev_runtime
                 AND c.next_media_time_start <= c.next_runtime
            THEN 12 -- ts / diff-show / fillable partially next + prev
            WHEN c.prevnextlive_sessdur_check = 'TS'
                 AND c.show_id_check = 'Diff'
                 AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
            THEN 13 -- ts / diff-show / fillable from prev
        END
      -- === Diff-station Live ===
      WHEN c.station_check = 'Diff' AND c.prevnextlive_sessdur_check = 'Live'
           AND c.show_id_check = 'Diff' AND c.prev_next_diff_2_second THEN
        CASE
            WHEN c.next_media_time_start - c.sess_dur >= 0
                 AND c.next_media_time_start <= c.next_runtime
            THEN 1  -- fillable from next
            WHEN c.next_media_time_start > 0
                 AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                 AND c.next_media_time_start <= c.next_runtime
            THEN 2  -- fillable partially next + prev
            WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
            THEN 3  -- fillable from prev
        END
      -- === Diff-station TS ===
      WHEN c.station_check = 'Diff' AND c.prevnextlive_sessdur_check = 'TS'
           AND c.show_id_check = 'Diff' AND c.prev_next_diff_2_second THEN
        CASE
            WHEN c.next_media_time_start - sess_dur >= 0
                 AND c.next_media_time_start <= next_runtime
            THEN 8  -- fillable from next
            WHEN c.next_media_time_start > 0
                 AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                 AND c.next_media_time_start <= next_runtime
            THEN 9  -- fillable partially next + prev
            WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
            THEN 10 -- fillable from prev
        END
    END AS metadata_fillable
FROM vcf AS c
WHERE c.station_check = 'Same'
   OR (c.station_check = 'Diff' AND c.prevnextlive_sessdur_check = 'Live')
   OR (c.station_check = 'Diff' AND c.prevnextlive_sessdur_check = 'TS')

In [0]:
%sql
SELECT metadata_fillable, COUNT(*), SUM(sess_dur)/3600.0
FROM dev.mohit_gangwani.smoothing_nulls_phase3_step2
GROUP BY 1

In [0]:
%sql
SELECT fk_station_id is not null as detected_session, COUNT(*), SUM(sess_dur)/3600.0
FROM dev.mohit_gangwani.smoothing_nulls_phase3_step1
GROUP BY 1

In [0]:
# %sql
# SELECT * FROM dev.mohit_gangwani.smoothing_nulls_phase3_step2 LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_phase3_step3;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_phase3_step3 AS
WITH vcf AS (
    SELECT *,
    CASE WHEN metadata_fillable IN (1,4,8,11,15) THEN 'Next'
         WHEN metadata_fillable IN (3,6,7,10,13,14,16) THEN 'Prev'
     --     WHEN metadata_fillable IN (2,5,9,12) THEN 'Split'
    END AS fillable
    FROM dev.mohit_gangwani.smoothing_nulls_phase3_step2
    WHERE metadata_fillable IN (1,3,4,6,7,8,10,13,14,11,15,16)
)
SELECT vcf.fk_tvid
, CASE WHEN fillable = 'Next' THEN next_show_id
       WHEN fillable = 'Prev' THEN prev_show_id
  END AS fk_show_id
, CASE WHEN fillable = 'Next' THEN next_st_id
       WHEN fillable = 'Prev' THEN prev_st_id
  END AS fk_station_id
, CASE WHEN fillable = 'Next' THEN next_airdate
       WHEN fillable = 'Prev' THEN prev_airdate
  END AS airdate
, vcf.session_start
, vcf.session_end
, vcf.sess_dur AS session_duration
, CASE WHEN fillable = 'Next' THEN next_media_time_start - sess_dur
       WHEN fillable = 'Prev' THEN prev_media_time_end
  END AS media_time_start
, CASE WHEN fillable = 'Next' THEN next_media_time_start
       WHEN fillable = 'Prev' THEN prev_media_time_end + sess_dur
  END AS media_time_end
, CASE WHEN fillable = 'Next' THEN next_runtime
       WHEN fillable = 'Prev' THEN prev_runtime
  END AS runtime
, c.fk_frame_id
, c.fk_dma_id
, c.fk_zoo_id
, CASE WHEN fillable = 'Next' THEN next_content
       WHEN fillable = 'Prev' THEN prev_content
  END AS fk_content_id
, vcf.fk_input_source_id
, c.fk_location_id
, CASE WHEN fillable = 'Next' THEN next_sche_id
       WHEN fillable = 'Prev' THEN prev_sche_id
  END AS fk_schedule_id
, c.timezone
, c.confidence
, c.ump_id
, CASE WHEN fillable = 'Next' THEN next_is_live
       WHEN fillable = 'Prev' THEN prev_is_live
  END AS is_live
, c.batch_size
, c.file_ingested
, c.created_at
, c.audio_contri
, c.video_contri
, c.local_session_start
, c.local_session_end
, c.vizio_epg_airing
, c.vizio_epg_station
, c.vizio_epg_program
, c.tuner_channel_id
, c.tuner_schedule_id
, c.tuner_program_id
, CASE WHEN fillable = 'Next' THEN next_tms_st_id
       WHEN fillable = 'Prev' THEN prev_tms_st_id
  END AS tms_station_id
, CASE WHEN fillable = 'Next' THEN next_tms_show_id
       WHEN fillable = 'Prev' THEN prev_tms_show_id
  END AS tms_show_id
, CASE WHEN fillable = 'Next' THEN next_tms_airdate
       WHEN fillable = 'Prev' THEN prev_tms_airdate
  END AS tms_airdate
, c.tms_schedule_id
, c.tms_tuner_channel_id
, c.tms_tuner_schedule_id
, c.tms_tuner_program_id
, c.tuner_channel_number
, c.reported_input_source
, c.content_type
, c.partition_key
, c.input_file_name
, c.vizio_epg_airing_start
, c.tuner_airdate
, c.tms_tuner_airdate
FROM vcf
JOIN prod.detection.viewing_content_firehose AS c
  ON c.fk_tvid = vcf.fk_tvid
 AND c.session_start = vcf.session_start
 AND c.session_end = vcf.session_end
WHERE c.session_start >= CURRENT_DATE - 1
  AND c.session_start < CURRENT_DATE
  AND c.session_end > c.session_start
GROUP BY ALL;

In [0]:
%sql
SELECT COUNT(*) FROM dev.mohit_gangwani.smoothing_nulls_phase3_step3

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_phase3_step4;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_phase3_step4 AS
SELECT c.* FROM dev.mohit_gangwani.smoothing_nulls_phase3_step3 c
UNION ALL
SELECT vc.*
FROM prod.detection.viewing_content_firehose AS vc
LEFT JOIN dev.mohit_gangwani.smoothing_nulls_phase3_step3 AS filt
  ON vc.fk_tvid = filt.fk_tvid
 AND vc.session_start = filt.session_start
 AND vc.session_end = filt.session_end
WHERE filt.fk_tvid IS NULL
 AND vc.session_start >= CURRENT_DATE - 1
 AND vc.session_start < CURRENT_DATE
 AND vc.session_end > vc.session_start
ORDER BY fk_tvid, session_start;

In [0]:
%sql
-- Materialize filtered firehose once to avoid double-scan in the next cell
CREATE OR REPLACE TEMPORARY TABLE firehose_filtered AS
SELECT *
FROM prod.detection.viewing_content_firehose
WHERE session_start >= CURRENT_DATE - 1
  AND session_start < CURRENT_DATE
  AND partition_key >= CURRENT_DATE - 1
  AND partition_key <= CURRENT_DATE
  AND session_end > session_start;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_phase3_step4;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_phase3_step4 AS
-- Uses cached firehose_filtered from previous cell (single scan)
WITH grouped_sessions AS (
    -- Step 1: Identify consecutive null-station groups (column-pruned for shuffle)
    SELECT
        sum(new_group) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS group_id,
        fk_tvid,
        fk_show_id,
        fk_station_id,
        session_start,
        session_end,
        session_duration,
        media_time_start,
        media_time_end,
        runtime,
        fk_input_source_id,
        fk_content_id,
        is_live,
        fk_schedule_id,
        airdate,
        tms_show_id,
        tms_airdate,
        tms_station_id
    FROM (
        SELECT
            fk_tvid,
            fk_show_id,
            fk_station_id,
            session_start,
            session_end,
            session_duration,
            media_time_start,
            media_time_end,
            runtime,
            fk_input_source_id,
            fk_content_id,
            is_live,
            fk_schedule_id,
            airdate,
            tms_show_id,
            tms_airdate,
            tms_station_id,
            CASE WHEN fk_station_id IS NULL
                AND LAG(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) IS NULL
                AND fk_input_source_id = LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start)
                AND DATEDIFF(SECOND, LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start), session_start) <= 1
                THEN 0
                ELSE 1
            END AS new_group
        FROM firehose_filtered
    )
)
, part_one AS (
    -- Step 2: Aggregate groups and compute neighbor lookups
    SELECT *,
        LEAD(fk_input_source_id) OVER w AS next_input,
        LAG(fk_input_source_id)  OVER w AS prev_input,
        LEAD(fk_show_id) OVER w AS next_show_id,
        LAG(fk_show_id)  OVER w AS prev_show_id,
        LEAD(fk_station_id) OVER w AS next_st_id,
        LAG(fk_station_id)  OVER w AS prev_st_id,
        LEAD(fk_schedule_id) OVER w AS next_sche_id,
        LAG(fk_schedule_id)  OVER w AS prev_sche_id,
        LEAD(airdate) OVER w AS next_airdate,
        LAG(airdate)  OVER w AS prev_airdate,
        LEAD(sess_dur) OVER w AS next_sess_dur,
        LAG(sess_dur)  OVER w AS prev_sess_dur,
        LEAD(is_live) OVER w AS next_is_live,
        LAG(is_live)  OVER w AS prev_is_live,
        LEAD(media_time_start) OVER w AS next_media_time_start,
        LAG(media_time_end)    OVER w AS prev_media_time_end,
        LEAD(runtime) OVER w AS next_runtime,
        LAG(runtime)  OVER w AS prev_runtime,
        LEAD(fk_content_id) OVER w AS next_content,
        LAG(fk_content_id)  OVER w AS prev_content,
        LEAD(session_start) OVER w AS next_start,
        LAG(session_end)    OVER w AS prev_end,
        LEAD(tms_airdate) OVER w AS next_tms_airdate,
        LAG(tms_airdate)  OVER w AS prev_tms_airdate,
        LEAD(tms_station_id) OVER w AS next_tms_st_id,
        LAG(tms_station_id)  OVER w AS prev_tms_st_id,
        LEAD(tms_show_id) OVER w AS next_tms_show_id,
        LAG(tms_show_id)  OVER w AS prev_tms_show_id
    FROM (
        SELECT
            group_id,
            fk_tvid,
            MAX(fk_station_id) AS fk_station_id,
            MAX(fk_input_source_id) AS fk_input_source_id,
            MAX(fk_schedule_id) AS fk_schedule_id,
            MAX(airdate) AS airdate,
            MAX(fk_show_id) AS fk_show_id,
            MAX(runtime) AS runtime,
            MAX(fk_content_id) AS fk_content_id,
            MIN(session_start) AS session_start,
            MAX(session_end) AS session_end,
            MAX(is_live) AS is_live,
            MIN(media_time_start) AS media_time_start,
            MAX(media_time_end) AS media_time_end,
            MAX(tms_airdate) AS tms_airdate,
            MAX(tms_show_id) AS tms_show_id,
            MAX(tms_station_id) AS tms_station_id,
            SUM(session_duration) AS sess_dur,
            COUNT(*) AS n_sess
        FROM grouped_sessions
        GROUP BY group_id, fk_tvid
    )
    WINDOW w AS (PARTITION BY fk_tvid ORDER BY session_start)
)
, part_two AS (
    SELECT c.*, 
        CASE
        -- === Same-station ===
        WHEN station_check = 'Same' THEN
            CASE
                WHEN c.prevnextlive_sessdur_check = 'Live'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 7  -- live / same-show / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'Live'
                    AND c.show_id_check = 'Diff'
                    AND c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 4  -- live / diff-show / fillable from next
                -- WHEN c.prevnextlive_sessdur_check = 'Live'
                --     AND c.show_id_check = 'Diff'
                --     AND c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 5  -- live / diff-show / fillable partially next + prev
                WHEN c.prevnextlive_sessdur_check = 'Live'
                    AND c.show_id_check = 'Diff'
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 6  -- live / diff-show / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 14 -- ts / same-show / fast-forwarding / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 16 -- ts / same-show / rewinding / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                    AND c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 15 -- ts / same-show / rewinding / fillable from next
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Diff'
                    AND c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 11 -- ts / diff-show / fillable from next
                -- WHEN c.prevnextlive_sessdur_check = 'TS'
                --     AND c.show_id_check = 'Diff'
                --     AND c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 12 -- ts / diff-show / fillable partially next + prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Diff'
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 13 -- ts / diff-show / fillable from prev
            END
        -- === Diff-station Live ===
        WHEN c.station_check = 'Diff'
         AND c.prevnextlive_sessdur_check = 'Live'
         AND c.show_id_check = 'Diff' THEN
            CASE
                WHEN c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 1  -- fillable from next
                -- WHEN c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 2  -- fillable partially next + prev
                WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 3  -- fillable from prev
            END
        -- === Diff-station TS ===
        WHEN c.station_check = 'Diff'
         AND c.prevnextlive_sessdur_check = 'TS'
         AND c.show_id_check = 'Diff' THEN
            CASE
                WHEN c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 8  -- fillable from next
                -- WHEN c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 9  -- fillable partially next + prev
                WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 10 -- fillable from prev
            END
        END AS metadata_fillable
    FROM (
        SELECT c.*
        , CASE WHEN c.prev_st_id <> c.next_st_id THEN 'Diff'
               WHEN c.prev_st_id = c.next_st_id THEN 'Same'
          END AS station_check
        , CASE WHEN c.prev_is_live = FALSE
                and c.next_is_live = FALSE
                and c.sess_dur <= 300 THEN 'TS'
               WHEN c.prev_is_live = TRUE
                and c.next_is_live = TRUE
                and c.sess_dur <= 180 THEN 'Live'
          END AS prevnextlive_sessdur_check
        , CASE WHEN c.prev_show_id = c.next_show_id THEN 'Same'
               WHEN c.prev_show_id <> c.next_show_id THEN 'Diff'
          END AS show_id_check
        FROM part_one c
        WHERE c.fk_station_id IS NULL
          AND c.prev_input = c.next_input
          AND c.prev_input = c.fk_input_source_id
          AND c.prev_st_id is not null
          and c.next_st_id is not null
          AND ((c.prev_is_live = FALSE and c.next_is_live = FALSE and c.sess_dur <= 300)
              OR (c.prev_is_live = TRUE and c.next_is_live = TRUE and c.sess_dur <= 180))
          AND timestampdiff(second, c.prev_end, c.session_start) <= 2
          and timestampdiff(second, c.session_end, c.next_start) <= 2
          and not (date_trunc('hour', c.session_start) + interval '1 hour' = date_trunc('hour', c.session_end)
                  or (minute(c.session_end) = 59 and second(c.session_end) = 59)
                  or (minute(c.session_start) = 00 and second(c.session_start) = 00))
    ) AS c
)
SELECT c.fk_tvid
, CASE WHEN c.fk_show_id IS NOT NULL THEN c.fk_show_id
       WHEN fill.fillable = 'Next' THEN fill.next_show_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_show_id
  END AS fk_show_id
, CASE WHEN c.fk_station_id IS NOT NULL THEN c.fk_station_id
       WHEN fill.fillable = 'Next' THEN fill.next_st_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_st_id
  END AS fk_station_id
, CASE WHEN c.airdate IS NOT NULL THEN c.airdate
       WHEN fill.fillable = 'Next' THEN fill.next_airdate
       WHEN fill.fillable = 'Prev' THEN fill.prev_airdate
  END AS airdate
, c.session_start
, c.session_end
, c.session_duration
, CASE WHEN c.media_time_start IS NOT NULL THEN c.media_time_start
       WHEN fill.fillable = 'Next' THEN fill.next_media_time_start - fill.sess_dur
       WHEN fill.fillable = 'Prev' THEN fill.prev_media_time_end
  END AS media_time_start
, CASE WHEN c.media_time_end IS NOT NULL THEN c.media_time_end
       WHEN fill.fillable = 'Next' THEN fill.next_media_time_start
       WHEN fill.fillable = 'Prev' THEN fill.prev_media_time_end + fill.sess_dur
  END AS media_time_end
, CASE WHEN c.runtime IS NOT NULL THEN c.runtime
       WHEN fill.fillable = 'Next' THEN fill.next_runtime
       WHEN fill.fillable = 'Prev' THEN fill.prev_runtime
  END AS runtime
, c.fk_frame_id
, c.fk_dma_id
, c.fk_zoo_id
, CASE WHEN NVL(c.fk_content_id, 1) != 3468026 THEN c.fk_content_id
       WHEN fill.fillable = 'Next' THEN fill.next_content
       WHEN fill.fillable = 'Prev' THEN fill.prev_content
       ELSE c.fk_content_id
  END AS fk_content_id
, c.fk_input_source_id
, c.fk_location_id
, CASE WHEN c.fk_schedule_id IS NOT NULL THEN c.fk_schedule_id
       WHEN fill.fillable = 'Next' THEN fill.next_sche_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_sche_id
  END AS fk_schedule_id
, c.timezone
, c.confidence
, c.ump_id
, CASE WHEN c.is_live IS NOT NULL THEN c.is_live
       WHEN fill.fillable = 'Next' THEN fill.next_is_live
       WHEN fill.fillable = 'Prev' THEN fill.prev_is_live
  END AS is_live
, c.batch_size
, c.file_ingested
, c.created_at
, c.audio_contri
, c.video_contri
, c.local_session_start
, c.local_session_end
, c.vizio_epg_airing
, c.vizio_epg_station
, c.vizio_epg_program
, c.tuner_channel_id
, c.tuner_schedule_id
, c.tuner_program_id
, CASE WHEN c.tms_station_id IS NOT NULL THEN c.tms_station_id
       WHEN fill.fillable = 'Next' THEN fill.next_tms_st_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_tms_st_id
  END AS tms_station_id
, CASE WHEN c.tms_show_id IS NOT NULL THEN c.tms_show_id
       WHEN fill.fillable = 'Next' THEN fill.next_tms_show_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_tms_show_id
  END AS tms_show_id
, CASE WHEN c.tms_airdate IS NOT NULL THEN c.tms_airdate
       WHEN fill.fillable = 'Next' THEN fill.next_tms_airdate
       WHEN fill.fillable = 'Prev' THEN fill.prev_tms_airdate
  END AS tms_airdate
, c.tms_schedule_id
, c.tms_tuner_channel_id
, c.tms_tuner_schedule_id
, c.tms_tuner_program_id
, c.tuner_channel_number
, c.reported_input_source
, c.content_type
, c.partition_key
, c.input_file_name
, c.vizio_epg_airing_start
, c.tuner_airdate
, c.tms_tuner_airdate
FROM firehose_filtered AS c
LEFT JOIN (
    SELECT *
    , CASE WHEN metadata_fillable IN (1,4,8,11,15) THEN 'Next'
           WHEN metadata_fillable IN (3,6,7,10,13,14,16) THEN 'Prev'
        --     WHEN metadata_fillable IN (2,5,9,12) THEN 'Split'
      END AS fillable
    FROM part_two
    WHERE metadata_fillable IN (1,3,4,6,7,8,10,13,14,11,15,16)
) AS fill
  ON c.fk_tvid = fill.fk_tvid
 AND c.session_start = fill.session_start
 AND c.session_end = fill.session_end

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_phase3_step4;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_phase3_step4 AS
WITH grouped_sessions AS (
            SELECT
                sum(new_group) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS group_id,
                fk_tvid,
                fk_show_id,
                fk_station_id,
                session_start,
                session_end,
                session_duration,
                media_time_start,
                media_time_end,
                runtime,
                fk_input_source_id,
                fk_content_id,
                is_live,
                airdate,
                tms_show_id,
                tms_airdate,
                tms_station_id,
                content_type
            FROM (
                SELECT
                    fk_tvid,
                    fk_show_id,
                    fk_station_id,
                    session_start,
                    session_end,
                    session_duration,
                    media_time_start,
                    media_time_end,
                    runtime,
                    fk_input_source_id,
                    fk_content_id,
                    is_live,
                    airdate,
                    tms_show_id,
                    tms_airdate,
                    tms_station_id,
                    content_type,
                    CASE WHEN fk_station_id IS NULL
                        AND LAG(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) IS NULL
                        AND fk_input_source_id = LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start)
                        AND DATEDIFF(SECOND, LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start), session_start) <= 1
                        THEN 0
                        ELSE 1
                    END AS new_group
                FROM firehose_filtered
            ))
, part_one AS (
            SELECT *,
                LEAD(fk_input_source_id) OVER w AS next_input,
                LAG(fk_input_source_id)  OVER w AS prev_input,
                LEAD(fk_show_id) OVER w AS next_show_id,
                LAG(fk_show_id)  OVER w AS prev_show_id,
                LEAD(fk_station_id) OVER w AS next_st_id,
                LAG(fk_station_id)  OVER w AS prev_st_id,
                LEAD(airdate) OVER w AS next_airdate,
                LAG(airdate)  OVER w AS prev_airdate,
                LEAD(sess_dur) OVER w AS next_sess_dur,
                LAG(sess_dur)  OVER w AS prev_sess_dur,
                LEAD(is_live) OVER w AS next_is_live,
                LAG(is_live)  OVER w AS prev_is_live,
                LEAD(media_time_start) OVER w AS next_media_time_start,
                LAG(media_time_end)    OVER w AS prev_media_time_end,
                LEAD(runtime) OVER w AS next_runtime,
                LAG(runtime)  OVER w AS prev_runtime,
                LEAD(fk_content_id) OVER w AS next_content,
                LAG(fk_content_id)  OVER w AS prev_content,
                LEAD(session_start) OVER w AS next_start,
                LAG(session_end)    OVER w AS prev_end,
                LEAD(tms_airdate) OVER w AS next_tms_airdate,
                LAG(tms_airdate)  OVER w AS prev_tms_airdate,
                LEAD(tms_station_id) OVER w AS next_tms_st_id,
                LAG(tms_station_id)  OVER w AS prev_tms_st_id,
                LEAD(tms_show_id) OVER w AS next_tms_show_id,
                LAG(tms_show_id)  OVER w AS prev_tms_show_id,
                LEAD(content_type) OVER w AS next_content_type,
                LAG(content_type)  OVER w AS prev_content_type
            FROM (
                SELECT
                    group_id,
                    fk_tvid,
                    MAX(fk_station_id) AS fk_station_id,
                    MAX(fk_input_source_id) AS fk_input_source_id,
                    MAX(airdate) AS airdate,
                    MAX(fk_show_id) AS fk_show_id,
                    MAX(runtime) AS runtime,
                    MAX(fk_content_id) AS fk_content_id,
                    MIN(session_start) AS session_start,
                    MAX(session_end) AS session_end,
                    MAX(is_live) AS is_live,
                    MIN(media_time_start) AS media_time_start,
                    MAX(media_time_end) AS media_time_end,
                    MAX(tms_airdate) AS tms_airdate,
                    MAX(tms_show_id) AS tms_show_id,
                    MAX(tms_station_id) AS tms_station_id,
                    SUM(session_duration) AS sess_dur,
                    MAX(GREATEST(content_type, '')) AS content_type,
                    COUNT(*) AS n_sess
                FROM grouped_sessions
                GROUP BY group_id, fk_tvid
            )
            WINDOW w AS (PARTITION BY fk_tvid ORDER BY session_start)
        )
        , part_two AS (
            SELECT c.*, 
                CASE WHEN station_check = 'Same' THEN
                    CASE WHEN c.prevnextlive_sessdur_check = 'Live'
                            AND c.show_id_check = 'Same'
                            AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                            AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                        THEN 7  -- live / same-show / fillable from prev
                        WHEN c.prevnextlive_sessdur_check = 'Live'
                            AND c.show_id_check = 'Diff'
                            AND c.next_media_time_start - c.sess_dur >= 0
                            AND c.next_media_time_start <= c.next_runtime
                        THEN 4  -- live / diff-show / fillable from next
                        WHEN c.prevnextlive_sessdur_check = 'Live'
                            AND c.show_id_check = 'Diff'
                            AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                        THEN 6  -- live / diff-show / fillable from prev
                        WHEN c.prevnextlive_sessdur_check = 'TS'
                            AND c.show_id_check = 'Same'
                            AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                            AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                        THEN 14 -- ts / same-show / fast-forwarding / fillable from prev
                        WHEN c.prevnextlive_sessdur_check = 'TS'
                            AND c.show_id_check = 'Same'
                            AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                            AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                        THEN 16 -- ts / same-show / rewinding / fillable from prev
                        WHEN c.prevnextlive_sessdur_check = 'TS'
                            AND c.show_id_check = 'Same'
                            AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                            AND c.next_media_time_start - c.sess_dur >= 0
                            AND c.next_media_time_start <= c.next_runtime
                        THEN 15 -- ts / same-show / rewinding / fillable from next
                        WHEN c.prevnextlive_sessdur_check = 'TS'
                            AND c.show_id_check = 'Diff'
                            AND c.next_media_time_start - c.sess_dur >= 0
                            AND c.next_media_time_start <= c.next_runtime
                        THEN 11 -- ts / diff-show / fillable from next
                        WHEN c.prevnextlive_sessdur_check = 'TS'
                            AND c.show_id_check = 'Diff'
                            AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                        THEN 13 -- ts / diff-show / fillable from prev
                    END
                WHEN c.station_check = 'Diff'
                AND c.prevnextlive_sessdur_check = 'Live'
                AND c.show_id_check = 'Diff' THEN
                    CASE WHEN c.next_media_time_start - c.sess_dur >= 0
                            AND c.next_media_time_start <= c.next_runtime
                        THEN 1  -- fillable from next
                        WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                        THEN 3  -- fillable from prev
                    END
                WHEN c.station_check = 'Diff'
                AND c.prevnextlive_sessdur_check = 'TS'
                AND c.show_id_check = 'Diff' THEN
                    CASE
                        WHEN c.next_media_time_start - c.sess_dur >= 0
                            AND c.next_media_time_start <= c.next_runtime
                        THEN 8  -- fillable from next
                        WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                        THEN 10 -- fillable from prev
                    END
                END AS metadata_fillable
            FROM (
                SELECT c.*
                , CASE WHEN c.prev_st_id <> c.next_st_id THEN 'Diff'
                    WHEN c.prev_st_id = c.next_st_id THEN 'Same'
                END AS station_check
                , CASE WHEN c.prev_is_live = FALSE
                        and c.next_is_live = FALSE
                        and c.sess_dur <= 300 THEN 'TS'
                    WHEN c.prev_is_live = TRUE
                        and c.next_is_live = TRUE
                        and c.sess_dur <= 180 THEN 'Live'
                END AS prevnextlive_sessdur_check
                , CASE WHEN c.prev_show_id = c.next_show_id THEN 'Same'
                    WHEN c.prev_show_id <> c.next_show_id THEN 'Diff'
                END AS show_id_check
                FROM part_one c
                WHERE c.fk_station_id IS NULL
                AND c.prev_input = c.next_input
                AND c.prev_input = c.fk_input_source_id
                AND c.prev_st_id is not null
                and c.next_st_id is not null
                AND ((c.prev_is_live = FALSE and c.next_is_live = FALSE and c.sess_dur <= 300)
                    OR (c.prev_is_live = TRUE and c.next_is_live = TRUE and c.sess_dur <= 180))
                AND timestampdiff(second, c.prev_end, c.session_start) <= 2
                and timestampdiff(second, c.session_end, c.next_start) <= 2
                and not (date_trunc('hour', c.session_start) + interval '1 hour' = date_trunc('hour', c.session_end)
                        or (minute(c.session_end) = 59 and second(c.session_end) = 59)
                        or (minute(c.session_start) = 00 and second(c.session_start) = 00))
            ) AS c
        )
        SELECT c.fk_tvid
        , CASE WHEN c.fk_show_id IS NOT NULL THEN c.fk_show_id
            WHEN fill.fillable = 'Next' THEN fill.next_show_id
            WHEN fill.fillable = 'Prev' THEN fill.prev_show_id
        END AS fk_show_id
        , CASE WHEN c.fk_station_id IS NOT NULL THEN c.fk_station_id
            WHEN fill.fillable = 'Next' THEN fill.next_st_id
            WHEN fill.fillable = 'Prev' THEN fill.prev_st_id
        END AS fk_station_id
        , CASE WHEN c.airdate IS NOT NULL THEN c.airdate
            WHEN fill.fillable = 'Next' THEN fill.next_airdate
            WHEN fill.fillable = 'Prev' THEN fill.prev_airdate
        END AS airdate
        , c.session_start
        , c.session_end
        , c.session_duration
        , CASE WHEN c.media_time_start IS NOT NULL THEN c.media_time_start
            WHEN fill.fillable = 'Next' THEN fill.next_media_time_start - fill.sess_dur
            WHEN fill.fillable = 'Prev' THEN fill.prev_media_time_end
        END AS media_time_start
        , CASE WHEN c.media_time_end IS NOT NULL THEN c.media_time_end
            WHEN fill.fillable = 'Next' THEN fill.next_media_time_start
            WHEN fill.fillable = 'Prev' THEN fill.prev_media_time_end + fill.sess_dur
        END AS media_time_end
        , CASE WHEN c.runtime IS NOT NULL THEN c.runtime
            WHEN fill.fillable = 'Next' THEN fill.next_runtime
            WHEN fill.fillable = 'Prev' THEN fill.prev_runtime
        END AS runtime
        , c.fk_dma_id
        , c.fk_zoo_id
        , CASE WHEN NVL(c.fk_content_id, 1) != 3468026 THEN c.fk_content_id
            WHEN fill.fillable = 'Next' THEN fill.next_content
            WHEN fill.fillable = 'Prev' THEN fill.prev_content
            ELSE c.fk_content_id
        END AS fk_content_id
        , c.fk_input_source_id
        , c.fk_location_id
        , CASE WHEN c.is_live IS NOT NULL THEN c.is_live
            WHEN fill.fillable = 'Next' THEN fill.next_is_live
            WHEN fill.fillable = 'Prev' THEN fill.prev_is_live
        END AS is_live
        , c.file_ingested
        , c.vizio_epg_airing
        , c.vizio_epg_station
        , c.vizio_epg_program
        , c.tuner_channel_id
        , c.tuner_program_id
        , CASE WHEN c.tms_station_id IS NOT NULL THEN c.tms_station_id
            WHEN fill.fillable = 'Next' THEN fill.next_tms_st_id
            WHEN fill.fillable = 'Prev' THEN fill.prev_tms_st_id
        END AS tms_station_id
        , CASE WHEN c.tms_show_id IS NOT NULL THEN c.tms_show_id
            WHEN fill.fillable = 'Next' THEN fill.next_tms_show_id
            WHEN fill.fillable = 'Prev' THEN fill.prev_tms_show_id
        END AS tms_show_id
        , CASE WHEN c.tms_airdate IS NOT NULL THEN c.tms_airdate
            WHEN fill.fillable = 'Next' THEN fill.next_tms_airdate
            WHEN fill.fillable = 'Prev' THEN fill.prev_tms_airdate
        END AS tms_airdate
        , c.tms_tuner_channel_id
        , c.tms_tuner_program_id
        , c.tuner_channel_number
        , c.reported_input_source
        , CASE WHEN NVL(c.vizio_epg_station, c.vizio_epg_program) IS NOT NULL THEN c.content_type
            WHEN fill.fillable = 'Next' THEN NULLIF(fill.next_content_type, '')
            WHEN fill.fillable = 'Prev' THEN NULLIF(fill.prev_content_type, '')
            ELSE c.content_type
          END AS content_type
        , c.vizio_epg_airing_start
        , c.tuner_airdate
        , c.tms_tuner_airdate
FROM firehose_filtered AS c
        LEFT JOIN (
            SELECT *
            , CASE WHEN metadata_fillable IN (1,4,8,11,15) THEN 'Next'
                WHEN metadata_fillable IN (3,6,7,10,13,14,16) THEN 'Prev'
            END AS fillable
            FROM part_two
            WHERE metadata_fillable IN (1,3,4,6,7,8,10,13,14,11,15,16)
        ) AS fill
        ON c.fk_tvid = fill.fk_tvid
        AND c.session_start = fill.session_start
        AND c.session_end = fill.session_end

In [0]:
%sql
DROP TABLE IF EXISTS firehose_filtered;

In [0]:
%sql
SELECT vizio_epg_station IS NOT NULL AS wf_session
, fk_station_id IS NOT NULL AS detected_session
, COUNT(*)
, SUM(session_duration)/3600.0 AS ttl_duration
FROM dev.mohit_gangwani.smoothing_nulls_phase3_step4
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
SELECT content_type
, COUNT(*)
, SUM(session_duration)/3600.0 AS ttl_duration
FROM dev.mohit_gangwani.smoothing_nulls_phase3_step4
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT session_start_hour
, content_type
, COUNT(*)
-- , SUM(session_duration)/3600.0 AS ttl_duration
FROM prod.detection.viewing_content_golden
WHERE session_start_hour >= CURRENT_DATE
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
SELECT session_start_hour
, content_type
, COUNT(*)
-- , SUM(session_duration)/3600.0 AS ttl_duration
FROM dev.detection.viewing_content_golden
WHERE session_start_hour >= CURRENT_DATE
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
SELECT vizio_epg_station IS NOT NULL AS wf_session
, fk_station_id IS NOT NULL AS detected_session
, COUNT(*)
, SUM(session_duration)/3600.0 AS ttl_duration
FROM prod.detection.viewing_content_firehose a
WHERE a.session_start >= CURRENT_DATE - 1
    AND a.session_start < CURRENT_DATE
    AND a.partition_key >= CURRENT_DATE - 1
    AND a.partition_key <= CURRENT_DATE
    AND a.session_end > a.session_start
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
SELECT vizio_epg_station IS NOT NULL AS wf_session
, tuner_channel_number IS NOT NULL AS tuner_session
, fk_station_id IS NOT NULL AS detected_session
, COUNT(*)
, SUM(session_duration)/3600.0 AS ttl_duration
FROM dev.mohit_gangwani.smoothing_nulls_phase3_step4
GROUP BY 1, 2, 3
ORDER BY 1, 2

In [0]:
%sql
SELECT vizio_epg_station IS NOT NULL AS wf_session
, tuner_channel_number IS NOT NULL AS tuner_session
, fk_station_id IS NOT NULL AS detected_session
, COUNT(*)
, SUM(session_duration)/3600.0 AS ttl_duration
FROM prod.detection.viewing_content_firehose a
WHERE a.session_start >= CURRENT_DATE - 1
    AND a.session_start < CURRENT_DATE
    AND a.partition_key >= CURRENT_DATE - 1
    AND a.partition_key <= CURRENT_DATE
    AND a.session_end > a.session_start
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_phase3_step4;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_phase3_step4 AS
WITH viewing_content_firehose AS (
    SELECT a.*
    FROM prod.detection.viewing_content_firehose a
    WHERE a.session_start >= CURRENT_DATE - 1
      AND a.session_start < CURRENT_DATE
      AND a.partition_key >= CURRENT_DATE - 1
      AND a.partition_key <= CURRENT_DATE
      AND a.session_end > a.session_start
)
, part_one AS (
    with vcf as (
        select *
        , sum(new_group) over (partition by fk_tvid order by session_start) group_id
        from (
            select 
            a.fk_tvid,
            a.fk_show_id,
            a.fk_station_id,
            a.session_start,
            a.session_end,
            a.session_duration,
            a.media_time_start,
            a.media_time_end,
            a.runtime,
            a.fk_input_source_id,
            a.fk_content_id,
            a.is_live,
            a.fk_schedule_id,
            a.airdate,
            a.tms_show_id,
            a.tms_airdate,
            a.tms_station_id,
            case when a.fk_station_id is null
                and lag(a.fk_station_id) over (partition by a.fk_tvid order by a.session_start) is null
                and a.fk_input_source_id = lag(a.fk_input_source_id) over (partition by a.fk_tvid order by a.session_start)
                and datediff(second, lag(a.session_end) over (partition by a.fk_tvid order by a.session_start), a.session_start) <= 1
                -- only consecutive nulls merged, sess_gap <= 1
                then 0
                else 1
            end AS new_group
            from viewing_content_firehose a
        )
    )
    select *,
    lead(fk_input_source_id) over (partition by fk_tvid order by session_start) AS next_input,
    lag(fk_input_source_id) over (partition by fk_tvid order by session_start) AS prev_input,
    lead(fk_show_id) over (partition by fk_tvid order by session_start) AS next_show_id,
    lag(fk_show_id) over (partition by fk_tvid order by session_start) AS prev_show_id,
    lead(fk_station_id) over (partition by fk_tvid order by session_start) AS next_st_id,
    lag(fk_station_id) over (partition by fk_tvid order by session_start) AS prev_st_id,
    lead(fk_schedule_id) over (partition by fk_tvid order by session_start) AS next_sche_id,
    lag(fk_schedule_id) over (partition by fk_tvid order by session_start) AS prev_sche_id,
    lead(airdate) over (partition by fk_tvid order by session_start) AS next_airdate,
    lag(airdate) over (partition by fk_tvid order by session_start) AS prev_airdate,
    lead(sess_dur) over (partition by fk_tvid order by session_start) AS next_sess_dur,
    lag(sess_dur) over (partition by fk_tvid order by session_start) AS prev_sess_dur,
    lead(is_live) over (partition by fk_tvid order by session_start) AS next_is_live,
    lag(is_live) over (partition by fk_tvid order by session_start) AS prev_is_live,
    lead(media_time_start) over (partition by fk_tvid order by session_start) AS next_media_time_start,
    -- lag(media_time_start) over (partition by fk_tvid order by session_start) AS prev_media_time_start,
    -- lead(media_time_end) over (partition by fk_tvid order by session_start) AS next_media_time_end,
    lag(media_time_end) over (partition by fk_tvid order by session_start) AS prev_media_time_end,
    lead(runtime) over (partition by fk_tvid order by session_start) AS next_runtime,
    lag(runtime) over (partition by fk_tvid order by session_start) AS prev_runtime,
    lead(fk_content_id) over (partition by fk_tvid order by session_start) AS next_content,
    lag(fk_content_id) over (partition by fk_tvid order by session_start) AS prev_content,
    lead(session_start) over (partition by fk_tvid order by session_start) AS next_start,
    lag(session_end) over (partition by fk_tvid order by session_start) AS prev_end,
    lead(tms_airdate) over (partition by fk_tvid order by session_start) AS next_tms_airdate,
    lag(tms_airdate) over (partition by fk_tvid order by session_start) AS prev_tms_airdate,
    lead(tms_station_id) over (partition by fk_tvid order by session_start) AS next_tms_st_id,
    lag(tms_station_id) over (partition by fk_tvid order by session_start) AS prev_tms_st_id,
    lead(tms_show_id) over (partition by fk_tvid order by session_start) AS next_tms_show_id,
    lag(tms_show_id) over (partition by fk_tvid order by session_start) AS prev_tms_show_id
    from (
        select 
            group_id,
            fk_tvid,
            max(fk_station_id) AS fk_station_id,
            max(fk_input_source_id) AS fk_input_source_id, 
            max(fk_schedule_id) AS fk_schedule_id,
            max(airdate) AS airdate,
            max(fk_show_id) AS fk_show_id,
            max(runtime) AS runtime,
            max(fk_content_id) AS fk_content_id,
            min(session_start) AS session_start,
            max(session_end) AS session_end,
            max(is_live) AS is_live,
            min(media_time_start) AS media_time_start,
            max(media_time_end) AS media_time_end,
            max(tms_airdate) AS tms_airdate,
            max(tms_show_id) AS tms_show_id,
            max(tms_station_id) AS tms_station_id,
            sum(session_duration) AS sess_dur,
            count(*) n_sess
        from vcf
        group by 1,2
    )
)
, part_two AS (
    SELECT c.*, 
        CASE
        -- === Same-station ===
        WHEN station_check = 'Same' THEN
            CASE
                WHEN c.prevnextlive_sessdur_check = 'Live'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 7  -- live / same-show / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'Live'
                    AND c.show_id_check = 'Diff'
                    AND c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 4  -- live / diff-show / fillable from next
                -- WHEN c.prevnextlive_sessdur_check = 'Live'
                --     AND c.show_id_check = 'Diff'
                --     AND c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 5  -- live / diff-show / fillable partially next + prev
                WHEN c.prevnextlive_sessdur_check = 'Live'
                    AND c.show_id_check = 'Diff'
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 6  -- live / diff-show / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur <= c.next_media_time_start
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 14 -- ts / same-show / fast-forwarding / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 16 -- ts / same-show / rewinding / fillable from prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Same'
                    AND c.prev_media_time_end + c.sess_dur > c.next_media_time_start
                    AND c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 15 -- ts / same-show / rewinding / fillable from next
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Diff'
                    AND c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 11 -- ts / diff-show / fillable from next
                -- WHEN c.prevnextlive_sessdur_check = 'TS'
                --     AND c.show_id_check = 'Diff'
                --     AND c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 12 -- ts / diff-show / fillable partially next + prev
                WHEN c.prevnextlive_sessdur_check = 'TS'
                    AND c.show_id_check = 'Diff'
                    AND c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 13 -- ts / diff-show / fillable from prev
            END
        -- === Diff-station Live ===
        WHEN c.station_check = 'Diff'
         AND c.prevnextlive_sessdur_check = 'Live'
         AND c.show_id_check = 'Diff' THEN
            CASE
                WHEN c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 1  -- fillable from next
                -- WHEN c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 2  -- fillable partially next + prev
                WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 3  -- fillable from prev
            END
        -- === Diff-station TS ===
        WHEN c.station_check = 'Diff'
         AND c.prevnextlive_sessdur_check = 'TS'
         AND c.show_id_check = 'Diff' THEN
            CASE
                WHEN c.next_media_time_start - c.sess_dur >= 0
                    AND c.next_media_time_start <= c.next_runtime
                THEN 8  -- fillable from next
                -- WHEN c.next_media_time_start > 0
                --     AND c.prev_media_time_end + (c.sess_dur - c.next_media_time_start) <= c.prev_runtime
                --     AND c.next_media_time_start <= c.next_runtime
                -- THEN 9  -- fillable partially next + prev
                WHEN c.prev_media_time_end + c.sess_dur <= c.prev_runtime
                THEN 10 -- fillable from prev
            END
        END AS metadata_fillable
    FROM (
        SELECT c.*
        , CASE WHEN c.prev_st_id <> c.next_st_id THEN 'Diff'
               WHEN c.prev_st_id = c.next_st_id THEN 'Same'
          END AS station_check
        , CASE WHEN c.prev_is_live = FALSE
                and c.next_is_live = FALSE
                and c.sess_dur <= 300 THEN 'TS'
               WHEN c.prev_is_live = TRUE
                and c.next_is_live = TRUE
                and c.sess_dur <= 180 THEN 'Live'
          END AS prevnextlive_sessdur_check
        , CASE WHEN c.prev_show_id = c.next_show_id THEN 'Same'
               WHEN c.prev_show_id <> c.next_show_id THEN 'Diff'
          END AS show_id_check
        FROM part_one c
        WHERE c.fk_station_id IS NULL
          AND c.prev_input = c.next_input
          AND c.prev_input = c.fk_input_source_id
          AND c.prev_st_id is not null
          and c.next_st_id is not null
          AND ((c.prev_is_live = FALSE and c.next_is_live = FALSE and c.sess_dur <= 300)
              OR (c.prev_is_live = TRUE and c.next_is_live = TRUE and c.sess_dur <= 180))
          AND timestampdiff(second, c.prev_end, c.session_start) <= 2
          and timestampdiff(second, c.session_end, c.next_start) <= 2
          and not (date_trunc('hour', c.session_start) + interval '1 hour' = date_trunc('hour', c.session_end)
                  or (minute(c.session_end) = 59 and second(c.session_end) = 59)
                  or (minute(c.session_start) = 00 and second(c.session_start) = 00))
    ) AS c
)
SELECT c.fk_tvid
, CASE WHEN c.fk_show_id IS NOT NULL THEN c.fk_show_id
       WHEN fill.fillable = 'Next' THEN fill.next_show_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_show_id
  END AS fk_show_id
, CASE WHEN c.fk_station_id IS NOT NULL THEN c.fk_station_id
       WHEN fill.fillable = 'Next' THEN fill.next_st_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_st_id
  END AS fk_station_id
, CASE WHEN c.airdate IS NOT NULL THEN c.airdate
       WHEN fill.fillable = 'Next' THEN fill.next_airdate
       WHEN fill.fillable = 'Prev' THEN fill.prev_airdate
  END AS airdate
, c.session_start
, c.session_end
, c.session_duration
, CASE WHEN c.media_time_start IS NOT NULL THEN c.media_time_start
       WHEN fill.fillable = 'Next' THEN fill.next_media_time_start - fill.sess_dur
       WHEN fill.fillable = 'Prev' THEN fill.prev_media_time_end
  END AS media_time_start
, CASE WHEN c.media_time_end IS NOT NULL THEN c.media_time_end
       WHEN fill.fillable = 'Next' THEN fill.next_media_time_start
       WHEN fill.fillable = 'Prev' THEN fill.prev_media_time_end + fill.sess_dur
  END AS media_time_end
, CASE WHEN c.runtime IS NOT NULL THEN c.runtime
       WHEN fill.fillable = 'Next' THEN fill.next_runtime
       WHEN fill.fillable = 'Prev' THEN fill.prev_runtime
  END AS runtime
, c.fk_frame_id
, c.fk_dma_id
, c.fk_zoo_id
, CASE WHEN NVL(c.fk_content_id, 1) != 3468026 THEN c.fk_content_id
       WHEN fill.fillable = 'Next' THEN fill.next_content
       WHEN fill.fillable = 'Prev' THEN fill.prev_content
       ELSE c.fk_content_id
  END AS fk_content_id
, c.fk_input_source_id
, c.fk_location_id
, CASE WHEN c.fk_schedule_id IS NOT NULL THEN c.fk_schedule_id
       WHEN fill.fillable = 'Next' THEN fill.next_sche_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_sche_id
  END AS fk_schedule_id
, c.timezone
, c.confidence
, c.ump_id
, CASE WHEN c.is_live IS NOT NULL THEN c.is_live
       WHEN fill.fillable = 'Next' THEN fill.next_is_live
       WHEN fill.fillable = 'Prev' THEN fill.prev_is_live
  END AS is_live
, c.batch_size
, c.file_ingested
, c.created_at
, c.audio_contri
, c.video_contri
, c.local_session_start
, c.local_session_end
, c.vizio_epg_airing
, c.vizio_epg_station
, c.vizio_epg_program
, c.tuner_channel_id
, c.tuner_schedule_id
, c.tuner_program_id
, CASE WHEN c.tms_station_id IS NOT NULL THEN c.tms_station_id
       WHEN fill.fillable = 'Next' THEN fill.next_tms_st_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_tms_st_id
  END AS tms_station_id
, CASE WHEN c.tms_show_id IS NOT NULL THEN c.tms_show_id
       WHEN fill.fillable = 'Next' THEN fill.next_tms_show_id
       WHEN fill.fillable = 'Prev' THEN fill.prev_tms_show_id
  END AS tms_show_id
, CASE WHEN c.tms_airdate IS NOT NULL THEN c.tms_airdate
       WHEN fill.fillable = 'Next' THEN fill.next_tms_airdate
       WHEN fill.fillable = 'Prev' THEN fill.prev_tms_airdate
  END AS tms_airdate
, c.tms_schedule_id
, c.tms_tuner_channel_id
, c.tms_tuner_schedule_id
, c.tms_tuner_program_id
, c.tuner_channel_number
, c.reported_input_source
, c.content_type
, c.partition_key
, c.input_file_name
, c.vizio_epg_airing_start
, c.tuner_airdate
, c.tms_tuner_airdate
FROM viewing_content_firehose AS c
LEFT JOIN (
    SELECT *
    , CASE WHEN metadata_fillable IN (1,4,8,11,15) THEN 'Next'
           WHEN metadata_fillable IN (3,6,7,10,13,14,16) THEN 'Prev'
        --     WHEN metadata_fillable IN (2,5,9,12) THEN 'Split'
      END AS fillable
    FROM part_two
    WHERE metadata_fillable IN (1,3,4,6,7,8,10,13,14,11,15,16)
) AS fill
  ON c.fk_tvid = fill.fk_tvid
 AND c.session_start = fill.session_start
 AND c.session_end = fill.session_end

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_comm_part_optimized;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_comm_part_optimized AS
WITH null_gaps AS (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
    , LEAD(session_start) OVER w AS next_start
    , LEAD(session_end) OVER w AS next_end
    , LAG(session_end) OVER w AS prev_end
    , LEAD(fk_show_id) OVER w AS next_show_id
    , LAG(fk_show_id) OVER w AS prev_show_id
    , LEAD(fk_station_id) OVER w AS next_station_id
    , LAG(fk_station_id) OVER w AS prev_station_id
    , LEAD(fk_input_source_id) OVER w AS next_input_source_id
    , LAG(fk_input_source_id) OVER w AS prev_input_source_id
    , LEAD(media_time_start) OVER w AS next_media_time_start
    , LAG(media_time_end) OVER w AS prev_media_time_end
    , LEAD(runtime) OVER w AS next_runtime
    , LAG(runtime) OVER w AS prev_runtime
    , LEAD(airdate) OVER w AS next_airdate
    , LAG(airdate) OVER w AS prev_airdate
    , LEAD(tms_airdate) OVER w AS next_tms_airdate
    , LAG(tms_airdate) OVER w AS prev_tms_airdate
    , LEAD(tms_station_id) OVER w AS next_tms_station_id
    , LAG(tms_station_id) OVER w AS prev_tms_station_id
    , LEAD(tms_show_id) OVER w AS next_tms_show_id
    , LAG(tms_show_id) OVER w AS prev_tms_show_id
    , LEAD(fk_content_id) OVER w AS next_fk_content_id
    , LAG(fk_content_id) OVER w AS prev_fk_content_id
    , LEAD(content_type) OVER w AS next_content_type
    , LAG(content_type) OVER w AS prev_content_type
    , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
    , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
    , (prev_end <= prev_airdate_end) AS prev_is_live
    , (next_end <= next_airdate_end) AS next_is_live
    FROM firehose_filtered
    WINDOW w AS (PARTITION BY fk_tvid ORDER BY session_start)
),
null_calc AS (
    SELECT *, 
        CASE 
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
            WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL THEN 3    -- Hot fix (Found Bug)
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
                THEN CASE WHEN NVL(next_media_time_start, 0) - session_duration >= 0
                            AND next_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                            AND NVL(next_is_live, FALSE) THEN 3
                        WHEN NVL(prev_media_time_end, 0) + session_duration <= NVL(prev_runtime, 0)
                        AND prev_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                        AND NVL(prev_is_live, FALSE) THEN 2
                    END
        END AS num_for_calc
    FROM null_gaps
    WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND next_start >= session_end
    AND prev_end <= session_start
    AND CASE
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
            THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
            AND prev_media_time_end + session_duration <= next_media_time_start
            AND next_is_live
            AND prev_is_live
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
            THEN prev_input_source_id <=> fk_input_source_id
            AND prev_media_time_end + session_duration <= prev_runtime
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND prev_is_live
        WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
            THEN next_input_source_id <=> fk_input_source_id
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
            AND next_media_time_start - session_duration >= 0
            AND next_is_live
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
            THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                        THEN prev_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                            AND prev_is_live
                    WHEN next_media_time_start - session_duration >= 0
                        THEN next_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                        AND next_is_live
                    ELSE FALSE END
        ELSE FALSE END
),
null_gaps_to_fill AS (
    SELECT 
        fk_tvid
        , session_start
        , session_end
        , session_duration
        , fk_input_source_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end
            WHEN num_for_calc = 3 THEN next_media_time_start - session_duration
        END AS media_time_start
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end + session_duration
            WHEN num_for_calc = 3 THEN next_media_time_start
        END AS media_time_end
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_station_id
            WHEN num_for_calc = 3 THEN next_station_id
        END AS fk_station_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_show_id
            WHEN num_for_calc = 3 THEN next_show_id
        END AS fk_show_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_airdate
            WHEN num_for_calc = 3 THEN next_airdate
        END AS airdate
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_runtime
            WHEN num_for_calc = 3 THEN next_runtime
        END AS runtime
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_airdate
            WHEN num_for_calc = 3 THEN next_tms_airdate
        END AS tms_airdate
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_station_id
            WHEN num_for_calc = 3 THEN next_tms_station_id
        END AS tms_station_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_show_id
            WHEN num_for_calc = 3 THEN next_tms_show_id
        END AS tms_show_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_fk_content_id
            WHEN num_for_calc = 3 THEN next_fk_content_id
        END AS fk_content_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_content_type
            WHEN num_for_calc = 3 THEN next_content_type
        END AS content_type
        , TRUE AS is_live
    FROM null_calc
            ),
comms AS (
    SELECT 
        vc.fk_tvid, 
        vc.session_start, 
        vc.session_end
    FROM prod.detection.viewing_commercials_firehose AS vc
    JOIN null_gaps_to_fill AS tv
        ON tv.fk_tvid = vc.fk_tvid
        AND vc.session_start <= tv.session_end
        AND vc.session_end >= tv.session_start
    WHERE vc.partition_key >= DATE(CURRENT_DATE - 1)
        AND vc.partition_key <= DATE(CURRENT_DATE)
        AND vc.session_start >= (SELECT MIN(session_start) - INTERVAL '1 HOUR' FROM null_gaps_to_fill)
        AND vc.session_start <= (SELECT MAX(session_end) + INTERVAL '1 HOUR' FROM null_gaps_to_fill)
    GROUP BY ALL
),
comm_duration AS (
    SELECT 
        fk_tvid, 
        tv_group_num
        , MIN(session_start) AS session_start
        , MAX(session_end) AS session_end
        , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
    FROM (
        SELECT 
            *, 
            SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
        FROM (
            SELECT fk_tvid, session_start, session_end
            , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
            , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
            FROM comms
    )
    )
    GROUP BY 1, 2
),
filled_null_gaps AS( 
    SELECT fk_tvid, session_start, session_end, session_duration, fk_input_source_id
    , media_time_start, media_time_end, is_live
    , runtime, airdate, fk_show_id, fk_station_id, tms_airdate, tms_show_id, tms_station_id, fk_content_id
    , content_type
    , SUM(comm_duration)*1.0 AS total_comm_duration
    FROM (
        SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration, content.fk_input_source_id
        , content.media_time_start, content.media_time_end, content.is_live, content.fk_show_id, content.airdate
        , content.runtime
        , content.fk_station_id, content.tms_airdate, content.tms_show_id, content.tms_station_id, content.fk_content_id
        , content.content_type
        , GREATEST(comms.session_start, content.session_start) AS comm_start
        , LEAST(comms.session_end, content.session_end) AS comm_end
        , TIMESTAMPDIFF(SECOND, comm_start, comm_end) AS comm_duration
        FROM null_gaps_to_fill AS content
        JOIN comm_duration comms
            ON content.fk_tvid = comms.fk_tvid
            AND comms.session_start <= content.session_end
            AND comms.session_end >= content.session_start
        WHERE content.fk_show_id IS NOT NULL
            AND content.airdate IS NOT NULL
            AND content.media_time_start IS NOT NULL
            AND content.fk_station_id IS NOT NULL
    )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17
),
filled_null_gaps_filter AS (
    SELECT 
        *
    FROM filled_null_gaps
    WHERE (session_duration <= 15 AND total_comm_duration >= 5)
       OR (session_duration > 15 AND total_comm_duration / session_duration >= 0.5)
)
SELECT 
    c.fk_tvid
    , c.session_start
    , c.session_end
    , COALESCE(n.session_duration, c.session_duration) AS session_duration
    , COALESCE(n.fk_show_id, c.fk_show_id) AS fk_show_id
    , COALESCE(n.fk_station_id, c.fk_station_id) AS fk_station_id
    , COALESCE(n.airdate, c.airdate) AS airdate
    , COALESCE(n.tms_airdate, c.tms_airdate) AS tms_airdate
    , COALESCE(n.tms_show_id, c.tms_show_id) AS tms_show_id
    , COALESCE(n.tms_station_id, c.tms_station_id) AS tms_station_id
    , COALESCE(n.media_time_start, c.media_time_start) AS media_time_start
    , COALESCE(n.media_time_end, c.media_time_end) AS media_time_end
    , COALESCE(n.runtime, c.runtime) AS runtime
    , COALESCE(n.fk_content_id, c.fk_content_id) AS fk_content_id
    , c.fk_input_source_id
    , c.fk_dma_id
    , c.fk_location_id
    , COALESCE(n.is_live, c.is_live) AS is_live
    , c.fk_zoo_id
    , c.tms_tuner_channel_id
    , c.tms_tuner_program_id
    , c.tms_tuner_airdate
    , c.tuner_channel_number
    , c.tuner_channel_id
    , c.tuner_program_id
    , c.tuner_airdate
    , c.vizio_epg_program
    , c.vizio_epg_station
    , c.vizio_epg_airing
    , c.vizio_epg_airing_start
    , c.reported_input_source
    , CASE WHEN COALESCE(c.vizio_epg_program, c.vizio_epg_station) IS NOT NULL THEN c.content_type
            ELSE n.content_type
        END AS content_type
    , c.file_ingested
FROM firehose_filtered c
LEFT JOIN filled_null_gaps_filter n
ON n.fk_tvid = c.fk_tvid
AND n.session_start = c.session_start
AND n.session_end = c.session_end
AND n.fk_input_source_id <=> c.fk_input_source_id;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.smoothing_nulls_comm_part_current;
CREATE OR REPLACE TABLE dev.mohit_gangwani.smoothing_nulls_comm_part_current AS
WITH null_gaps AS (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
        , LEAD(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
        , LEAD(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
        , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
        , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
        , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
        , LEAD(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_station_id
        , LAG(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_station_id
        , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
        , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
        , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
        , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
        , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
        , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
        , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
        , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
        , LEAD(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_airdate
        , LAG(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_airdate
        , LEAD(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_station_id
        , LAG(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_station_id
        , LEAD(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_show_id
        , LAG(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_show_id
        , LEAD(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_fk_content_id
        , LAG(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_fk_content_id
        , LEAD(content_type) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_content_type
        , LAG(content_type) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_content_type
        , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
        , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
        , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
        , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
        FROM firehose_filtered
),
null_calc AS (
    SELECT *, 
        CASE 
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
            WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL THEN 3    -- Hot fix (Found Bug)
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
                THEN CASE WHEN NVL(next_media_time_start, 0) - session_duration >= 0
                            AND next_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                            AND NVL(next_is_live, FALSE) THEN 3
                            WHEN NVL(prev_media_time_end, 0) + session_duration <= NVL(prev_runtime, 0)
                            AND prev_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                            AND NVL(prev_is_live, FALSE) THEN 2
                    END
        END AS num_for_calc
    FROM null_gaps
    WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND next_start >= session_end
    AND prev_end <= session_start
    AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
        THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
        AND prev_media_time_end + session_duration <= next_media_time_start
        AND next_is_live
        AND prev_is_live
        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
        AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
        THEN prev_input_source_id <=> fk_input_source_id
        AND prev_media_time_end + session_duration <= prev_runtime
        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
        AND prev_is_live
        WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
        THEN next_input_source_id <=> fk_input_source_id
        AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        AND next_media_time_start - session_duration >= 0
        AND next_is_live
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
        THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                    THEN prev_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                        AND prev_is_live
                    WHEN next_media_time_start - session_duration >= 0
                    THEN next_input_source_id <=> fk_input_source_id
                    AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                    AND next_is_live
                    ELSE FALSE END
        ELSE FALSE END
    GROUP BY ALL

),
null_gaps_to_fill AS (
    SELECT 
        fk_tvid
        , session_start
        , session_end
        , session_duration
        , fk_input_source_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end
            WHEN num_for_calc = 3 THEN next_media_time_start - session_duration
        END AS media_time_start
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end + session_duration
            WHEN num_for_calc = 3 THEN next_media_time_start
        END AS media_time_end
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_station_id
            WHEN num_for_calc = 3 THEN next_station_id
        END AS fk_station_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_show_id
            WHEN num_for_calc = 3 THEN next_show_id
        END AS fk_show_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_airdate
            WHEN num_for_calc = 3 THEN next_airdate
        END AS airdate
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_runtime
            WHEN num_for_calc = 3 THEN next_runtime
        END AS runtime
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_airdate
            WHEN num_for_calc = 3 THEN next_tms_airdate
        END AS tms_airdate
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_station_id
            WHEN num_for_calc = 3 THEN next_tms_station_id
        END AS tms_station_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_show_id
            WHEN num_for_calc = 3 THEN next_tms_show_id
        END AS tms_show_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_fk_content_id
            WHEN num_for_calc = 3 THEN next_fk_content_id
        END AS fk_content_id
        , CASE WHEN num_for_calc IN (1, 2) THEN prev_content_type
            WHEN num_for_calc = 3 THEN next_content_type
        END AS content_type
        , TRUE AS is_live
        , fk_input_source_id
    FROM null_calc
    GROUP BY ALL
),
comms AS (
    SELECT 
        vc.fk_tvid, 
        vc.session_start, 
        vc.session_end
    FROM detection.viewing_commercials_firehose AS vc
    JOIN null_gaps_to_fill AS tv
        ON tv.fk_tvid = vc.fk_tvid
        AND vc.session_start <= tv.session_end
        AND vc.session_end >= tv.session_start
    WHERE partition_key >= DATE(CURRENT_DATE - 1)
        AND partition_key <= DATE(CURRENT_DATE)
    GROUP BY ALL
    ORDER BY 1, 2, 3
),
comm_duration AS (
    SELECT 
        fk_tvid, 
        tv_group_num
        , MIN(session_start) AS session_start
        , MAX(session_end) AS session_end
        , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
    FROM (
        SELECT 
        *, 
        SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
        SELECT fk_tvid, session_start, session_end
        , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
        , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
        FROM comms
    )
    )
    GROUP BY 1, 2
),
filled_null_gaps AS ( 
    SELECT fk_tvid, session_start, session_end, session_duration, fk_input_source_id
    , media_time_start, media_time_end, is_live
    , runtime, airdate, fk_show_id, fk_station_id, tms_airdate, tms_show_id, tms_station_id, fk_content_id
    , content_type
    , SUM(comm_duration)*1.0 AS total_comm_duration
    FROM (
    SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration, content.fk_input_source_id
    , content.media_time_start, content.media_time_end, content.is_live, content.fk_show_id, content.airdate
    , content.runtime, content.fk_input_source_id
    , content.fk_station_id, content.tms_airdate, content.tms_show_id, content.tms_station_id, content.fk_content_id
    , content.content_type
    , GREATEST(comms.session_start, content.session_start) AS comm_start
    , LEAST(comms.session_end, content.session_end) AS comm_end
    , TIMESTAMPDIFF(SECOND, comm_start, comm_end) AS comm_duration
    FROM null_gaps_to_fill AS content
    JOIN comm_duration comms
        ON content.fk_tvid = comms.fk_tvid
        AND comms.session_start <= content.session_end
        AND comms.session_end >= content.session_start
    WHERE content.fk_show_id IS NOT NULL
        AND content.airdate IS NOT NULL
        AND content.media_time_start IS NOT NULL
        AND content.fk_station_id IS NOT NULL
    )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17
),
filled_null_gaps_filter AS (
    SELECT  *
    FROM filled_null_gaps
    WHERE 
        CASE WHEN session_duration <= 15 
            THEN total_comm_duration >= 5
        ELSE total_comm_duration/session_duration >= 0.5 
    END
)
SELECT 
    c.fk_tvid
    , c.session_start
    , c.session_end
    , COALESCE(n.session_duration, c.session_duration) AS session_duration
    , COALESCE(n.fk_show_id, c.fk_show_id) AS fk_show_id
    , COALESCE(n.fk_station_id, c.fk_station_id) AS fk_station_id
    , COALESCE(n.airdate, c.airdate) AS airdate
    , COALESCE(n.tms_airdate, c.tms_airdate) AS tms_airdate
    , COALESCE(n.tms_show_id, c.tms_show_id) AS tms_show_id
    , COALESCE(n.tms_station_id, c.tms_station_id) AS tms_station_id
    , COALESCE(n.media_time_start, c.media_time_start) AS media_time_start
    , COALESCE(n.media_time_end, c.media_time_end) AS media_time_end
    , COALESCE(n.runtime, c.runtime) AS runtime
    , COALESCE(n.fk_content_id, c.fk_content_id) AS fk_content_id
    , c.fk_input_source_id
    , c.fk_dma_id
    , c.fk_location_id
    , COALESCE(n.is_live, c.is_live) AS is_live
    , c.fk_zoo_id
    , c.tms_tuner_channel_id
    , c.tms_tuner_program_id
    , c.tms_tuner_airdate
    , c.tuner_channel_number
    , c.tuner_channel_id
    , c.tuner_program_id
    , c.tuner_airdate
    , c.vizio_epg_program
    , c.vizio_epg_station
    , c.vizio_epg_airing
    , c.vizio_epg_airing_start
    , c.reported_input_source
    , CASE WHEN COALESCE(c.vizio_epg_program, c.vizio_epg_station) IS NOT NULL THEN c.content_type
            ELSE n.content_type
        END AS content_type
    , c.file_ingested
FROM firehose_filtered c
LEFT JOIN filled_null_gaps_filter n
ON n.fk_tvid = c.fk_tvid
AND n.session_start = c.session_start
AND n.session_end = c.session_end
AND n.fk_input_source_id = c.fk_input_source_id
ORDER BY 1, 2, 3

In [0]:
%sql
SELECT vizio_epg_station IS NOT NULL AS wf_session
, tuner_channel_number IS NOT NULL AS tuner_session
, fk_station_id IS NOT NULL AS detected_session
, COUNT(*)
, SUM(session_duration)/3600.0 AS ttl_duration
FROM dev.mohit_gangwani.smoothing_nulls_comm_part_optimized
GROUP BY 1, 2, 3
ORDER BY 1, 2

In [0]:
%sql
SELECT vizio_epg_station IS NOT NULL AS wf_session
, tuner_channel_number IS NOT NULL AS tuner_session
, fk_station_id IS NOT NULL AS detected_session
, COUNT(*)
, SUM(session_duration)/3600.0 AS ttl_duration
FROM dev.mohit_gangwani.smoothing_nulls_comm_part_current
GROUP BY 1, 2, 3
ORDER BY 1, 2

In [0]:
%sql
SELECT * FROM prod.public.cid_map
WHERE customer = 'videoamp'
  AND value = 'Pudding Plate :15'
LIMIT 100

In [0]:
%sql
-- Estimate filtered epg_show size per hourly batch
-- Shows distinct show IDs (union of 4 join columns) and estimated filtered table size per hour
WITH hourly_ids AS (
    SELECT DATE_TRUNC('hour', session_start) AS hour_bucket, tms_tuner_program_id AS id
    FROM dev.mohit_gangwani.smoothing_nulls_comm_part_optimized WHERE tms_tuner_program_id IS NOT NULL
    UNION
    SELECT DATE_TRUNC('hour', session_start), tuner_program_id
    FROM dev.mohit_gangwani.smoothing_nulls_comm_part_optimized WHERE tuner_program_id IS NOT NULL
    UNION
    SELECT DATE_TRUNC('hour', session_start), tms_show_id
    FROM dev.mohit_gangwani.smoothing_nulls_comm_part_optimized WHERE tms_show_id IS NOT NULL
    UNION
    SELECT DATE_TRUNC('hour', session_start), fk_show_id
    FROM dev.mohit_gangwani.smoothing_nulls_comm_part_optimized WHERE fk_show_id IS NOT NULL
)
SELECT 
    hour_bucket,
    COUNT(DISTINCT id) AS distinct_show_ids,
    ROUND(COUNT(DISTINCT id) * 1.0 / (SELECT COUNT(*) FROM prod.detection.epg_show) * 782, 1) AS est_filtered_size_mb,
    CASE WHEN ROUND(COUNT(DISTINCT id) * 1.0 / (SELECT COUNT(*) FROM prod.detection.epg_show) * 782, 1) <= 10 
         THEN 'YES - auto-broadcast'
         ELSE 'NO - needs hint or threshold bump'
    END AS under_10mb_threshold
FROM hourly_ids
GROUP BY hour_bucket
ORDER BY hour_bucket

In [0]:
import requests
from dateutil import parser as dtparser

host = "https://dbc-a931e1c8-6b6d.cloud.databricks.com"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}"}
cluster_id = "0303-213057-cf92n5zl"
app_id = "app-20260812235344-0000"

run1_start, run1_end = 1786632618659, 1786633201424
run2_start, run2_end = 1786636218378, 1786636882598

resp = requests.get(
    "{}/driver-proxy-api/o/2719976097125283/{}/40001/api/v1/applications/{}/stages".format(host, cluster_id, app_id),
    headers=headers
)
stages = resp.json()

def get_run_metrics(stages, start_ms, end_ms):
    run_stages = []
    for stage in stages:
        if stage.get('status') == 'SKIPPED':
            continue
        sub_time_str = stage.get('submissionTime', '')
        if sub_time_str:
            sub_time = dtparser.parse(sub_time_str)
            sub_epoch_ms = sub_time.timestamp() * 1000
            if start_ms <= sub_epoch_ms <= end_ms:
                run_stages.append(stage)
    return {
        'stages': len(run_stages),
        'shuffle_read': sum(s.get('shuffleReadBytes', 0) for s in run_stages),
        'shuffle_write': sum(s.get('shuffleWriteBytes', 0) for s in run_stages),
        'shuffle_read_rec': sum(s.get('shuffleReadRecords', 0) for s in run_stages),
        'shuffle_write_rec': sum(s.get('shuffleWriteRecords', 0) for s in run_stages),
        'disk_spill': sum(s.get('diskBytesSpilled', 0) for s in run_stages),
        'mem_spill': sum(s.get('memoryBytesSpilled', 0) for s in run_stages),
        'input_bytes': sum(s.get('inputBytes', 0) for s in run_stages),
        'output_bytes': sum(s.get('outputBytes', 0) for s in run_stages),
        'input_rec': sum(s.get('inputRecords', 0) for s in run_stages),
        'output_rec': sum(s.get('outputRecords', 0) for s in run_stages),
        'executor_runtime': sum(s.get('executorRunTime', 0) for s in run_stages),
        'total_tasks': sum(s.get('numCompleteTasks', 0) for s in run_stages),
    }

m1 = get_run_metrics(stages, run1_start, run1_end)
m2 = get_run_metrics(stages, run2_start, run2_end)

GB = 1024**3

# Print as a structured dict for clean formatting
results = {
    "run_14759163010261_smoothing": {
        "execution_duration": "581s",
        "stages": m1['stages'],
        "total_tasks": m1['total_tasks'],
        "executor_cpu_time_s": int(m1['executor_runtime'] / 1000),
        "shuffle_read_gb": round(m1['shuffle_read'] / GB, 2),
        "shuffle_read_records": m1['shuffle_read_rec'],
        "shuffle_write_gb": round(m1['shuffle_write'] / GB, 2),
        "shuffle_write_records": m1['shuffle_write_rec'],
        "disk_spill_gb": round(m1['disk_spill'] / GB, 2),
        "memory_spill_gb": round(m1['mem_spill'] / GB, 2),
        "input_gb": round(m1['input_bytes'] / GB, 2),
        "input_records": m1['input_rec'],
        "output_gb": round(m1['output_bytes'] / GB, 2),
        "output_records": m1['output_rec'],
    },
    "run_117369513076725_master": {
        "execution_duration": "663s",
        "stages": m2['stages'],
        "total_tasks": m2['total_tasks'],
        "executor_cpu_time_s": int(m2['executor_runtime'] / 1000),
        "shuffle_read_gb": round(m2['shuffle_read'] / GB, 2),
        "shuffle_read_records": m2['shuffle_read_rec'],
        "shuffle_write_gb": round(m2['shuffle_write'] / GB, 2),
        "shuffle_write_records": m2['shuffle_write_rec'],
        "disk_spill_gb": round(m2['disk_spill'] / GB, 2),
        "memory_spill_gb": round(m2['mem_spill'] / GB, 2),
        "input_gb": round(m2['input_bytes'] / GB, 2),
        "input_records": m2['input_rec'],
        "output_gb": round(m2['output_bytes'] / GB, 2),
        "output_records": m2['output_rec'],
    },
    "delta_master_vs_smoothing": {
        "execution_duration": "+82s (+14.1%)",
        "stages_diff": m2['stages'] - m1['stages'],
        "shuffle_read_diff_gb": round((m2['shuffle_read'] - m1['shuffle_read']) / GB, 2),
        "shuffle_write_diff_gb": round((m2['shuffle_write'] - m1['shuffle_write']) / GB, 2),
        "disk_spill_diff_gb": round((m2['disk_spill'] - m1['disk_spill']) / GB, 2),
        "input_diff_gb": round((m2['input_bytes'] - m1['input_bytes']) / GB, 2),
        "output_diff_gb": round((m2['output_bytes'] - m1['output_bytes']) / GB, 2),
    }
}

import json
print(json.dumps(results, indent=2))